In [ ]:
from pdfquery import PDFQuery
import pandas as pd
import os
from functools import partial


def get_TRANSPORTE_CAMPO_1(pdf) -> str:
    """Obtiene el nombre de la empresa porteadora del archivo PDF usando pdfquery."""
    ETIQUETAS_A_IGNORAR = [
        "/ Nome e endereco do transportador",
        "Nome e endereco do transportador",
        "Nombre y domicilio del porteador",
    ]
    try:
        base = pdf.pq('LTTextLineHorizontal:contains("Nombre y domicilio del porteador")')
        if not base:
            return ""

        # eq(0) evita duplicados cuando se cargaron todas las páginas (MIC/DTA + CRT)
        siguiente = base.eq(0).next()

        for _ in range(6):
            if siguiente is None:
                break
            texto = siguiente.text().strip()

            if not texto or texto.isdigit():
                siguiente = siguiente.next()
                continue

            if any(pat in texto for pat in ETIQUETAS_A_IGNORAR):
                siguiente = siguiente.next()
                continue

            # Es el contenido real; tomar solo la primera línea (nombre de empresa)
            primera_linea = siguiente.find('LTTextLineHorizontal').eq(0)
            if len(primera_linea):
                return primera_linea.text().strip()
            return texto.splitlines()[0]

        return ""
    except KeyError as e:
        print(f"Error: {e}")
        return ""

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

def listar_archivos(directorio, formato=None):
    """
    Lista los archivos en un directorio dado con un formato especifico.
    Si no se especifica un formato, lista todos los archivos.
    Retorna las rutas absolutas.
    """
    directorio = os.path.abspath(directorio)  # asegura que el path sea absoluto
    archivos = []
    for file in os.listdir(directorio):
        path_completo = os.path.join(directorio, file)
        if os.path.isfile(path_completo) and (formato is None or file.lower().endswith(formato.lower())):
            archivos.append(path_completo)
    return archivos

listar_archivos_pdf = partial(listar_archivos, formato='.pdf') 
__all__ = ['listar_archivos_pdf'] 
   

In [ ]:
archivos = listar_archivos_pdf(r'C:\Rapha\repos\pdfs-to-excel-1\data\AG2026')
for archivo in archivos:

    pdf = PDFQuery(archivo)
    pdf.load()
    transporte_campo_1 = get_TRANSPORTE_CAMPO_1(pdf)
    print(f"{archivo} - {transporte_campo_1}")

In [7]:
archivo_viejo = r"C:\Rapha\repos\pdfs-to-excel-1\data\AG2025\25AR132590G.pdf"
archivo_nuevo = r"C:\Rapha\repos\pdfs-to-excel-1\data\AG2026\26AR052590X.pdf"
pdf = PDFQuery(archivo_nuevo)
pdf.load(0)

get_TRANSPORTE_CAMPO_1(pdf)

'TRANSPORTES CRUZ DEL SUR S.A.'